# Python mínimo para la Semana 2

En la Semana 1 viste Python base y pandas. Los cuadernos de esta semana usan tres herramientas nuevas que conviene manejar antes:

1. **numpy** — trabajar con vectores y matrices de números "de golpe" (sin bucles).
2. **matplotlib** — dibujar gráficas sencillas.
3. **scikit-learn** — el patrón `fit` / `predict` para entrenar y usar un modelo.

Cada bloque es corto: léelo, ejecuta las celdas y haz los **mini-retos**. No hace falta nada del proyecto de flotación; usamos ejemplos de juguete.

## 1. numpy: operar sobre muchos números a la vez

In [ ]:
import numpy as np

# Una lista de Python y un "array" de numpy con los mismos valores
silices_lista = [2.35, 1.80, 3.10, 2.05]
silices = np.array([2.35, 1.80, 3.10, 2.05])

print(silices)
tipo = type(silices)
print(tipo)

La ventaja del array: puedes operar sobre **todos** los valores a la vez, sin escribir un bucle. Esto se llama *vectorización*.

In [ ]:
doble = silices * 2                # cada valor multiplicado por 2
print(doble)

media = silices.mean()
centrado = silices - media         # a cada valor le resta la media
print(centrado)

Comparación: lo mismo **con bucle** (más largo y lento). En el curso preferimos la forma vectorizada de arriba.

In [ ]:
resultado = []
for v in silices_lista:
    resultado.append(v * 2)
print(resultado)

Y puedes operar **entre dos arrays**, posición a posición:

In [ ]:
iron = np.array([60.1, 62.0, 58.5, 61.2])
print(iron + silices)     # suma el 1º con el 1º, el 2º con el 2º, ...

> **Mini-reto.** Crea un array `caudal = np.array([390, 405, 380, 410])` y calcula, en una sola línea vectorizada, cada valor **dividido entre 100**.

In [ ]:
# Tu código aquí


## 2. Elegir elementos: indexado y troceado

In [ ]:
print(silices[0])     # el primero (las posiciones empiezan en 0)
print(silices[-1])    # el último
print(silices[:2])    # los dos primeros (troceado)

**Indexado booleano:** puedes quedarte solo con los valores que cumplen una condición. Primero se calcula una *máscara* de `True`/`False`, y luego se filtra con ella.

In [ ]:
mascara = silices > 3
print(mascara)              # [False False  True False]
print(silices[mascara])     # solo los que superan 3

> **Mini-reto.** Usando `iron`, imprime solo los valores **mayores o iguales que 61**.

In [ ]:
# Tu código aquí


## 3. Resúmenes y forma de los datos

In [ ]:
print("media :", silices.mean())
print("desv.  :", silices.std())
print("máximo :", silices.max())
print("mínimo :", silices.min())

# Una raíz cuadrada (la usarás para el RMSE): sqrt del error cuadrático medio
media = silices.mean()
desvios = silices - media
cuadrados = desvios**2
mse = np.mean(cuadrados)
raiz = np.sqrt(mse)
print("tipo RMSE:", raiz)

Cuando hay **varias columnas** (una matriz 2D), puedes resumir **por columna** con `axis=0`. Y `.shape` te dice (filas, columnas).

In [ ]:
M = np.array([[1.0, 60.0],
              [2.0, 62.0],
              [3.0, 58.0]])
print("forma:", M.shape)        # (3 filas, 2 columnas)
print("media por columna:", M.mean(axis=0))

## 4. Construir una matriz de diseño: `column_stack`

Un modelo lineal necesita una **matriz** donde cada columna es una variable. `np.column_stack` pega varios arrays como columnas; `np.ones` crea una columna de unos (para el término independiente).

In [ ]:
x1 = np.array([0.2, 0.5, 0.9])
x2 = np.array([50.0, 30.0, 42.0])
unos = np.ones(3)

X = np.column_stack([unos, x1, x2])   # columnas: [1s | x1 | x2]
print(X)
print("forma:", X.shape)

## 5. El producto matricial `@` (la pieza clave)

`X @ beta` combina cada **fila** de la matriz `X` con el vector de coeficientes `beta` y produce **una predicción por fila**. Es el corazón de la regresión lineal y del descenso de gradiente.

In [ ]:
beta = np.array([-2.0, 5.0, 20.0])   # un coeficiente por columna de X

pred = X @ beta                       # (3x3) @ (3,) -> (3,)
print(pred)

La regla de las formas: si `X` es `(n, p)` y `beta` es `(p,)`, entonces `X @ beta` es `(n,)` — un número por fila. Si las formas no encajan, numpy da error (es una pista útil para detectar fallos).

> **Mini-reto.** Con `X` de arriba y `beta2 = np.array([0.0, 1.0, 1.0])`, calcula `X @ beta2` e imprime el resultado. ¿Qué está sumando cada fila?

In [ ]:
# Tu código aquí


## 6. matplotlib: gráficas mínimas

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

horas = np.arange(1, 6)                 # 1,2,3,4,5
sil   = np.array([2.3, 1.9, 3.1, 2.6, 3.4])

plt.figure(figsize=(6, 3))
plt.plot(horas, sil, 'o-')              # 'o-' = puntos unidos por línea
plt.xlabel("hora")
plt.ylabel("% sílice")
plt.title("Ejemplo")
plt.grid(alpha=0.3)
plt.show()

Dos series y una leyenda; `scatter` dibuja solo puntos:

In [ ]:
real = np.array([2.3, 1.9, 3.1, 2.6, 3.4])
estim = np.array([2.5, 2.0, 2.9, 2.7, 3.1])

plt.figure(figsize=(6, 3))
plt.plot(horas, real, 'o-', label="real")
plt.plot(horas, estim, 's--', label="estimado")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## 7. scikit-learn: el patrón `fit` / `predict`

Casi todos los modelos de sklearn se usan igual: se **crea** el modelo, se **entrena** con `fit(X, y)` y se **usa** con `predict(X)`. Un detalle: sklearn quiere la `X` en 2D (una columna por variable); para una sola variable se usa `.reshape(-1, 1)`.

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score

x = np.array([1, 2, 3, 4, 5])
x = x.reshape(-1, 1)                    # 2D: 5 filas, 1 columna
y = np.array([2.1, 3.9, 6.2, 7.8, 10.1])       # ~ 2*x

modelo = LinearRegression()
modelo.fit(x, y)                # aprender
pred = modelo.predict(x)        # predecir

mae = mean_absolute_error(y, pred)
r2 = r2_score(y, pred)
print("predicciones:", pred.round(2))
print(f"MAE: {mae:.3f}   R²: {r2:.3f}")

**Estandarizar** (media 0, desviación 1) es habitual antes de entrenar. La regla de oro: el escalador se ajusta **solo con los datos de entrenamiento** y luego se aplica a todo. Y la **partición temporal** se hace troceando: primeras filas para entrenar, últimas para evaluar.

In [ ]:
from sklearn.preprocessing import StandardScaler

col1 = np.arange(10)
col2 = col1 * 0.5 + 1.0
X = np.column_stack([col1, col2])     # 10 filas, 2 columnas
k = int(len(X) * 0.8)                 # 80% para entrenar
Xtr = X[:k]                           # primeras filas (partición temporal)
Xte = X[k:]                           # últimas filas

scaler = StandardScaler()
scaler.fit(Xtr)                       # ajustar SOLO con train
Xtr_s = scaler.transform(Xtr)
Xte_s = scaler.transform(Xte)
print("train:", Xtr.shape, " test:", Xte.shape)

medias = Xtr_s.mean(axis=0)
print("media de cada columna en train tras estandarizar:", medias.round(3))

> **Mini-reto.** Con `x` e `y` del ejemplo de `LinearRegression`, entrena el modelo y predice el valor para `x = 6` (pista: crea primero `x_nuevo = np.array([[6]])` y después usa `modelo.predict(x_nuevo)`).

In [ ]:
# Tu código aquí


---
## Ya estás listo

Con esto puedes seguir los cuadernos de la semana:

- **numpy** para vectores, matrices, `column_stack` y el producto `@`,
- **matplotlib** para ver curvas de coste y ajustes,
- **scikit-learn** con `fit` / `predict`, `StandardScaler` y las métricas.

Lo demás (qué significa cada modelo y por qué funciona) se explica en el cuaderno satélite.